In [1]:
import pandas as pd

In [2]:
csv_file = "./confidence_rank.csv"

In [3]:
import pandas as pd

# Load your CSV file
df = pd.read_csv("confidence_rank.csv")

# Ensure all relevant columns are strings
for col in ["Regulator", "Target", "Model"]:
    df[col] = df[col].astype(str)

# ---------------------------------------------------------
# Build dictionary: node_2 -> models that originally contain it
# ---------------------------------------------------------

result_dict = {}

def extract_matches(col_name):
    matched_df = df[df[col_name].str.endswith("_2")]

    for key, group in matched_df.groupby(col_name):
        result_dict.setdefault(key, set()).update(group["Model"])

extract_matches("Regulator")
extract_matches("Target")

for key in result_dict:
    result_dict[key] = set(result_dict[key])   # keep as sets for fast lookup

print(result_dict)

# ---------------------------------------------------------
# Generate new edges
# ---------------------------------------------------------

new_rows = []

for _, row in df.iterrows():

    model = row["Model"]
    reg = row["Regulator"]
    tar = row["Target"]

    reg_variants = [reg]
    tar_variants = [tar]

    # Should we introduce reg_2?
    if not reg.endswith("_2"):
        reg2 = reg + "_2"
        if reg2 in result_dict and model not in result_dict[reg2]:
            reg_variants.append(reg2)

    # Should we introduce tar_2?
    if not tar.endswith("_2"):
        tar2 = tar + "_2"
        if tar2 in result_dict and model not in result_dict[tar2]:
            tar_variants.append(tar2)

    # Create every non-original combination
    for new_reg in reg_variants:
        for new_tar in tar_variants:

            if new_reg == reg and new_tar == tar:
                continue

            new_row = row.copy()
            new_row["Regulator"] = new_reg
            new_row["Target"] = new_tar

            # Check for A -> A_2 or A_2 -> A
            is_A_to_A2 = (
                not new_reg.endswith("_2")
                and new_tar.endswith("_2")
                and new_reg == new_tar.removesuffix("_2")
            )

            is_A2_to_A = (
                new_reg.endswith("_2")
                and not new_tar.endswith("_2")
                and new_reg.removesuffix("_2") == new_tar
            )

            if is_A_to_A2:
                new_row["Confidence Rank"] = "0"
                new_row["Constraint"] = "necessary"
                new_row["Direct"] = ""
                new_row["Notes"] = (
                    "Reaching a higher level necessitates achieving the lower level first."
                )

            elif is_A2_to_A:
                new_row["Confidence Rank"] = "-"
                new_row["Constraint"] = ""
                new_row["Direct"] = "wrong"
                new_row["Notes"] = (
                    "High-level should not activate low-level."
                )

            else:
                orig_note = "" if pd.isna(row["Notes"]) else str(row["Notes"])

                new_row["Notes"] = (
                    f"Duplicated from original edge "
                    f"({reg} -> {tar}, Model={model}). "
                    f"Original note: {orig_note}"
                )

            new_rows.append(new_row)

# ---------------------------------------------------------
# Combine with original dataframe if desired
# ---------------------------------------------------------

new_edges = pd.DataFrame(new_rows)

# Optional: remove duplicates
new_edges = new_edges.drop_duplicates()

# Final dataframe
df_augmented = pd.concat([df, new_edges], ignore_index=True).drop_duplicates()

print(f"Original edges : {len(df)}")
print(f"New edges      : {len(new_edges)}")
print(f"Total edges    : {len(df_augmented)}")

{'IFNGR_2': {'M06'}, 'IFNG_2': {'M06'}, 'IL12RB1_2': {'N10'}, 'IL2R_2': {'AJ', 'N10'}, 'IL4RA_2': {'N10'}, 'IL4R_2': {'N10'}, 'STAT1_2': {'M06'}, 'STAT5_2': {'AJ', 'N10'}, 'TBET_2': {'M06'}}
Original edges : 691
New edges      : 230
Total edges    : 921


In [4]:
# Save only the newly created edges
new_edges.to_csv("confidence_rank_new_edges.csv", index=False)

In [5]:
# Count total edges
n_original = len(df)
n_new = len(new_edges)
n_total = len(df_augmented)

# Count unique regulator-target pairs
n_unique_original = len(df[["Regulator", "Target"]].drop_duplicates())
n_unique_new = len(new_edges[["Regulator", "Target"]].drop_duplicates())
n_unique_total = len(df_augmented[["Regulator", "Target"]].drop_duplicates())

print(f"Original edges          : {n_original}")
print(f"New edges               : {n_new}")
print(f"Total edges             : {n_total}")
print()
print(f"Unique original edges   : {n_unique_original}")
print(f"Unique new edges        : {n_unique_new}")
print(f"Unique total edges      : {n_unique_total}")

Original edges          : 691
New edges               : 230
Total edges             : 921

Unique original edges   : 395
Unique new edges        : 101
Unique total edges      : 473


In [6]:
cols_to_check = ["Sign", "Confidence Rank", "Constraint", "Direct"]

conflicts = []

for (reg, tar), group in df_augmented.groupby(["Regulator", "Target"]):
    differing = {}

    for col in cols_to_check:
        values = group[col].dropna().unique()
        if len(values) > 1:
            differing[col] = list(values)

    if differing:
        conflicts.append({
            "Regulator": reg,
            "Target": tar,
            "Conflicting columns": differing
        })

print(f"Found {len(conflicts)} regulator-target pairs with inconsistencies.\n")

for conflict in conflicts:
    print(f"{conflict['Regulator']} -> {conflict['Target']}")
    for col, values in conflict["Conflicting columns"].items():
        print(f"  {col}: {values}")
    print()

Found 6 regulator-target pairs with inconsistencies.

IFNG -> TBET
  Sign: ['negative', 'positive']
  Confidence Rank: ['-', '2']
  Direct: ['wrong', 'indirect']

IFNG -> TBET_2
  Sign: ['negative', 'positive']
  Confidence Rank: ['-', '2']
  Direct: ['wrong', 'indirect']

IFNG_2 -> TBET
  Sign: ['negative', 'positive']
  Confidence Rank: ['-', '2']
  Direct: ['wrong', 'indirect']

IFNG_2 -> TBET_2
  Sign: ['negative', 'positive']
  Confidence Rank: ['-', '2']
  Direct: ['wrong', 'indirect']

NFKB -> IL2
  Sign: ['ambiguous', 'positive']

STAT1_2 -> TBET
  Sign: ['negative', 'positive']
  Confidence Rank: ['-', '1']
  Direct: ['wrong', 'direct']

